In [ ]:
# ======================================================
# Create Production Structure
# ======================================================

from pathlib import Path


ROOT = Path.cwd()


folders = [

    "src",

    "app",

    "data/input",

    "data/processed",

    "data/output",

    "temp",

    "config"

]


for folder in folders:

    (ROOT/folder).mkdir(

        parents=True,

        exist_ok=True

    )


print("Project Structure Created")

In [ ]:
config_code = """

from pathlib import Path


ROOT = Path(__file__).parent.parent


DATA = ROOT / "data"


INPUT_DIR = DATA / "input"


PROCESSED_DIR = DATA / "processed"


OUTPUT_DIR = DATA / "output"



CONFIG = {


    "ocr_languages":

    ["th","en"],


    "target_similarity":

    0.99,


    "max_iterations":

    20,


    "unit":

    "mm"


}

"""


(ROOT/"src/config.py").write_text(

    config_code,

    encoding="utf-8"

)


print("config.py created")

In [ ]:
loader_code = """

import cv2


def load_image(path):


    img=cv2.imread(

        str(path)

    )


    if img is None:

        raise Exception(
            "Cannot load image"
        )


    return img



"""


(ROOT/"src/loader.py").write_text(

    loader_code,

    encoding="utf-8"

)


print("loader.py created")

In [ ]:
preprocess_code = """

import cv2



def preprocess(img):


    gray=cv2.cvtColor(

        img,

        cv2.COLOR_BGR2GRAY

    )


    clean=cv2.fastNlMeansDenoising(

        gray

    )


    binary=cv2.adaptiveThreshold(

        clean,

        255,

        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,

        cv2.THRESH_BINARY,

        31,

        5

    )


    return {


        "gray":gray,


        "binary":binary


    }


"""


(ROOT/"src/preprocessing.py").write_text(

    preprocess_code,

    encoding="utf-8"

)


print("preprocessing.py created")

In [ ]:
ocr_code = """

import easyocr



reader = easyocr.Reader(

    ["th","en"],

    gpu=True

)



def detect_text(image):


    result = reader.readtext(

        image

    )


    return result



"""


(ROOT/"src/ocr_engine.py").write_text(

    ocr_code,

    encoding="utf-8"

)


print("ocr_engine.py created")

In [ ]:
geometry_code = """

import cv2



def extract_geometry(edge):


    lines=cv2.HoughLinesP(

        edge,

        1,

        3.14159/180,

        80

    )


    objects=[]


    if lines is not None:


        for l in lines:


            x1,y1,x2,y2=l[0]


            objects.append({


            "type":"LINE",


            "start":[x1,y1],


            "end":[x2,y2]


            })


    return objects



"""


(ROOT/"src/geometry_ai.py").write_text(

    geometry_code,

    encoding="utf-8"

)


print("geometry_ai.py created")

In [ ]:
dxf_code = """

import ezdxf



def create_dxf(objects,path):


    doc=ezdxf.new()


    msp=doc.modelspace()


    for obj in objects:


        if obj["type"]=="LINE":


            msp.add_line(

                obj["start"],

                obj["end"]

            )



    doc.saveas(path)



"""


(ROOT/"src/dxf_writer.py").write_text(

    dxf_code,

    encoding="utf-8"

)


print("dxf_writer.py created")

In [ ]:
optimizer_code = """

def optimize(score,target=0.99):


    if score>=target:

        return True


    return False



"""


(ROOT/"src/optimizer.py").write_text(

    optimizer_code,

    encoding="utf-8"

)


print("optimizer.py created")

In [ ]:
main_code = """

from src.loader import load_image

from src.preprocessing import preprocess

from src.geometry_ai import extract_geometry

from src.dxf_writer import create_dxf



def run(input_file,output_file):


    img=load_image(

        input_file

    )


    result=preprocess(

        img

    )


    edges=result["binary"]



    geometry=extract_geometry(

        edges

    )


    create_dxf(

        geometry,

        output_file

    )


    return output_file




if __name__=="__main__":


    run(

    "data/input/test.png",

    "data/output/result.dxf"

    )



"""


(ROOT/"main.py").write_text(

    main_code,

    encoding="utf-8"

)


print("main.py created")

In [ ]:
requirements = """

opencv-python
numpy
ezdxf
easyocr
scikit-image
gradio
torch
torchvision

"""


(ROOT/"requirements.txt").write_text(

    requirements,

    encoding="utf-8"

)


print("requirements created")

In [ ]:
import sys

sys.path.append(
    str(ROOT)
)


from src.preprocessing import preprocess


print(
"Production Import OK"
)